In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly as py
import seaborn as sns
from pandasql import sqldf
import plotly.graph_objects as go

In [35]:
pysqldf = lambda q: sqldf(q, globals())

In [36]:
df = pd.read_csv("video-3_analysis.csv")

In [37]:
df.head()

,frame_number,timestamp,ball_x,ball_y,player_1_x_meter,player_1_y_meter,player_2_x_meter,player_2_y_meter,is_rally_frame,rally_id,is_wall_hit,wall_hit_x_meter,wall_hit_y_meter,is_racket_hit,racket_hit_player_id,stroke_type,shot_type,shot_direction,shot_depth
0,0,0.000000,895.000000,419.000000,2.011459,7.675627,5.885906,6.113835,True,0,False,NaN,NaN,False,-1,NaN,NaN,NaN,NaN
1,1,0.016667,895.000000,419.000000,2.011459,7.675627,5.885906,6.113835,True,0,False,NaN,NaN,False,-1,NaN,NaN,NaN,NaN
2,2,0.033333,895.000000,419.000000,2.011459,7.675627,5.885906,6.113835,True,0,False,NaN,NaN,False,-1,NaN,NaN,NaN,NaN
3,3,0.050000,893.096774,418.451613,2.011459,7.675627,5.885906,6.113835,True,0,False,NaN,NaN,False,-1,NaN,NaN,NaN,NaN
4,4,0.066667,891.193548,417.903226,2.011459,7.675627,5.885906,6.113835,True,0,False,NaN,NaN,False,-1,NaN,NaN,NaN,NaN


## **Feature Engineering**

### Add: Player In T-Zone

In [38]:
# Define T-zone parameters (standard squash court)
T_X = 3.2  # meters (half of 6.4m court width)
T_Y = 5.44  # CHECKKKKKKKKKKKKKKKKKKKKKKKkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
T_RADIUS = 1.2  # meters

# Sort the dataframe and modify it directly (no copy)
df = df.sort_values(['rally_id', 'timestamp'])

# Calculate distance from T for each player at each timestamp
df['p1_distance_from_t'] = np.sqrt(
    (df['player_1_x_meter'] - T_X)**2 + 
    (df['player_1_y_meter'] - T_Y)**2
)

df['p2_distance_from_t'] = np.sqrt(
    (df['player_2_x_meter'] - T_X)**2 + 
    (df['player_2_y_meter'] - T_Y)**2
)

# Mark if player is in T-zone
df['p1_in_t_zone'] = df['p1_distance_from_t'] <= T_RADIUS
df['p2_in_t_zone'] = df['p2_distance_from_t'] <= T_RADIUS


## **1. Distance Covered Per Rally**

In [39]:
# Calculate distance moved between consecutive frames for each player
df_sorted = df.sort_values(['rally_id', 'timestamp'])

# Player 1 distance calculation
df_sorted['p1_distance'] = np.sqrt(
    (df_sorted['player_1_x_meter'].diff())**2 + 
    (df_sorted['player_1_y_meter'].diff())**2
)

# Player 2 distance calculation
df_sorted['p2_distance'] = np.sqrt(
    (df_sorted['player_2_x_meter'].diff())**2 + 
    (df_sorted['player_2_y_meter'].diff())**2
)

# Reset distance to 0 when rally changes (diff() carries over between rallies)
df_sorted.loc[df_sorted['rally_id'] != df_sorted['rally_id'].shift(), 'p1_distance'] = 0
df_sorted.loc[df_sorted['rally_id'] != df_sorted['rally_id'].shift(), 'p2_distance'] = 0

# Group by rally and sum distances
rally_coverage = df_sorted.groupby('rally_id').agg({
    'p1_distance': 'sum',
    'p2_distance': 'sum'
}).reset_index()

rally_coverage.columns = ['rally_id', 'p1_total_distance', 'p2_total_distance']

print(rally_coverage)

# Visualization - comparison of coverage per rally using Plotly
fig = go.Figure()

fig.add_trace(go.Bar(
    x=rally_coverage['rally_id'],
    y=rally_coverage['p1_total_distance'],
    name='Player 1',
    marker_color='#636EFA',
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=rally_coverage['rally_id'],
    y=rally_coverage['p2_total_distance'],
    name='Player 2',
    marker_color='#EF553B',
    opacity=0.8
))

fig.update_layout(
    title='Court Coverage per Rally',
    xaxis_title='Rally ID',
    yaxis_title='Distance Covered (meters)',
    barmode='group',
    template='plotly_white',
    height=500,
    width=1000
)

fig.show()

# Summary statistics
print(f"\nPlayer 1 - Avg coverage per rally: {rally_coverage['p1_total_distance'].mean():.2f}m")
print(f"Player 2 - Avg coverage per rally: {rally_coverage['p2_total_distance'].mean():.2f}m")

    rally_id  p1_total_distance  p2_total_distance
0          0          42.527145          46.773752
1          1          37.217019          31.411779
2          2          20.021049          22.099697
3          3          31.086469          23.503480
4          4          16.502991          22.666054
5          5          30.847769          31.673875
6          6          29.805032          33.828794
7          7          59.413022          66.018107
8          8          15.602055          15.787714
9          9          34.880081          30.252156
10        10          26.675473          30.485723



Player 1 - Avg coverage per rally: 31.33m
Player 2 - Avg coverage per rally: 32.23m


**Interaction:** Select one or both players

### **SQL Queries**

####  Total per rally

In [40]:
rally_id = None
player_id = None
start_time = None
end_time = None

court_coverage_query = """
WITH ordered_data AS (
  SELECT 
    rally_id,
    timestamp,
    player_1_x_meter,
    player_1_y_meter,
    player_2_x_meter,
    player_2_y_meter,
    LAG(player_1_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_x,
    LAG(player_1_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_y,
    LAG(player_2_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_x,
    LAG(player_2_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_y
  FROM df
  WHERE 1=1
),
distances AS (
  SELECT
    rally_id,
    CASE 
      WHEN prev_p1_x IS NOT NULL THEN
        SQRT(POWER(player_1_x_meter - prev_p1_x, 2) + POWER(player_1_y_meter - prev_p1_y, 2))
      ELSE 0
    END AS p1_distance,
    CASE 
      WHEN prev_p2_x IS NOT NULL THEN
        SQRT(POWER(player_2_x_meter - prev_p2_x, 2) + POWER(player_2_y_meter - prev_p2_y, 2))
      ELSE 0
    END AS p2_distance
  FROM ordered_data
)
SELECT
  rally_id,
  SUM(p1_distance) AS p1_total_distance,
  SUM(p2_distance) AS p2_total_distance
FROM distances
GROUP BY rally_id
ORDER BY rally_id
"""

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    # Filter to show only selected player's data (both columns still returned but can filter post-query)
    conditions.append(f"1=1")  # Player filtering applied post-aggregation
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

# Add conditions to the WHERE clause in the CTE
if conditions:
    court_coverage_query = court_coverage_query.replace("WHERE 1=1", "WHERE " + " AND ".join(conditions))

court_coverage_result = pysqldf(court_coverage_query)
print(court_coverage_result)

    rally_id  p1_total_distance  p2_total_distance
0          0          42.527145          46.773752
1          1          37.217019          31.411779
2          2          20.021049          22.099697
3          3          31.086469          23.503480
4          4          16.502991          22.666054
5          5          30.847769          31.673875
6          6          29.805032          33.828794
7          7          59.413022          66.018107
8          8          15.602055          15.787714
9          9          34.880081          30.252156
10        10          26.675473          30.485723


####  Average per player

In [41]:
rally_id = None
player_id = None
start_time = None
end_time = None

avg_distance_all_rallies_query = """
WITH ordered_data AS (
    SELECT
        rally_id,
        timestamp,
        player_1_x_meter,
        player_1_y_meter,
        player_2_x_meter,
        player_2_y_meter,
        LAG(player_1_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_x,
        LAG(player_1_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_y,
        LAG(player_2_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_x,
        LAG(player_2_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_y
    FROM df
    WHERE 1=1
),
distances AS (
    SELECT
        rally_id,
        CASE 
            WHEN prev_p1_x IS NOT NULL THEN
                SQRT(POWER(player_1_x_meter - prev_p1_x, 2) + POWER(player_1_y_meter - prev_p1_y, 2))
            ELSE 0
        END AS p1_distance,
        CASE 
            WHEN prev_p2_x IS NOT NULL THEN
                SQRT(POWER(player_2_x_meter - prev_p2_x, 2) + POWER(player_2_y_meter - prev_p2_y, 2))
            ELSE 0
        END AS p2_distance
    FROM ordered_data
),
rally_totals AS (
    SELECT
        rally_id,
        SUM(p1_distance) AS p1_total_distance,
        SUM(p2_distance) AS p2_total_distance
    FROM distances
    GROUP BY rally_id
)
SELECT
    ROUND(AVG(p1_total_distance), 2) AS player_1_avg_distance,
    ROUND(AVG(p2_total_distance), 2) AS player_2_avg_distance
FROM rally_totals;
"""

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")

if player_id is not None:
    # Player filter applied post-aggregation if needed
    conditions.append("1=1")

if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")

if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

if conditions:
    avg_distance_all_rallies_query = avg_distance_all_rallies_query.replace(
        "WHERE 1=1", "WHERE " + " AND ".join(conditions)
    )

avg_distance_all_rallies_result = pysqldf(avg_distance_all_rallies_query)
print(avg_distance_all_rallies_result)


   player_1_avg_distance  player_2_avg_distance
0                  31.33                  32.23


## **2. Total Distance Covered**

In [42]:
# Calculate distance moved between consecutive frames for each player
df_sorted = df.sort_values(['rally_id', 'timestamp'])

# Player 1 distance calculation
df_sorted['p1_distance'] = np.sqrt(
    (df_sorted['player_1_x_meter'].diff())**2 + 
    (df_sorted['player_1_y_meter'].diff())**2
)

# Player 2 distance calculation
df_sorted['p2_distance'] = np.sqrt(
    (df_sorted['player_2_x_meter'].diff())**2 + 
    (df_sorted['player_2_y_meter'].diff())**2
)

# Reset distance to 0 when rally changes (diff() carries over between rallies)
df_sorted.loc[df_sorted['rally_id'] != df_sorted['rally_id'].shift(), 'p1_distance'] = 0
df_sorted.loc[df_sorted['rally_id'] != df_sorted['rally_id'].shift(), 'p2_distance'] = 0

# Sum all distances across all rallies
p1_total_distance = df_sorted['p1_distance'].sum()
p2_total_distance = df_sorted['p2_distance'].sum()

print(f"Player 1 - Total distance covered: {p1_total_distance:.2f}m")
print(f"Player 2 - Total distance covered: {p2_total_distance:.2f}m")


Player 1 - Total distance covered: 344.58m
Player 2 - Total distance covered: 354.50m


### **SQL Query**

In [43]:
# SQL Query with optional filters

rally_id = None
player_id = None
start_time = None
end_time = None

distance_covered_query = """
WITH ordered_data AS (
  SELECT 
    rally_id,
    timestamp,
    player_1_x_meter,
    player_1_y_meter,
    player_2_x_meter,
    player_2_y_meter,
    LAG(player_1_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_x,
    LAG(player_1_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p1_y,
    LAG(player_2_x_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_x,
    LAG(player_2_y_meter) OVER (PARTITION BY rally_id ORDER BY timestamp) AS prev_p2_y
  FROM df
  WHERE 1=1
),
distances AS (
  SELECT
    rally_id,
    CASE 
      WHEN prev_p1_x IS NOT NULL THEN
        SQRT(POWER(player_1_x_meter - prev_p1_x, 2) + POWER(player_1_y_meter - prev_p1_y, 2))
      ELSE 0
    END AS p1_distance,
    CASE 
      WHEN prev_p2_x IS NOT NULL THEN
        SQRT(POWER(player_2_x_meter - prev_p2_x, 2) + POWER(player_2_y_meter - prev_p2_y, 2))
      ELSE 0
    END AS p2_distance
  FROM ordered_data
)
SELECT
  ROUND(SUM(p1_distance), 2) AS p1_total_distance,
  ROUND(SUM(p2_distance), 2) AS p2_total_distance
FROM distances
"""

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"1=1")  # Player filtering handled in result display
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

# Add conditions to the WHERE clause in the CTE
if conditions:
    distance_covered_query = distance_covered_query.replace("WHERE 1=1", "WHERE " + " AND ".join(conditions))

distance_covered_result = pysqldf(distance_covered_query)
print(distance_covered_result)

   p1_total_distance  p2_total_distance
0             344.58              354.5


## **3. Time To T**

In [44]:
#### Time-to-T (Fixed - No Double Counting)

import plotly.graph_objects as go
import pandas as pd

# Get all racket hit events sorted by time
racket_hits = df[df['is_racket_hit'] == True].sort_values(['rally_id', 'timestamp']).copy()

# Calculate time-to-T for each shot
time_to_t_results = []

for idx, hit in racket_hits.iterrows():
    player_id = hit['racket_hit_player_id']
    rally_id = hit['rally_id']
    hit_timestamp = hit['timestamp']
    
    # Find this player's NEXT shot in the same rally
    next_shots = racket_hits[
        (racket_hits['rally_id'] == rally_id) &
        (racket_hits['racket_hit_player_id'] == player_id) &
        (racket_hits['timestamp'] > hit_timestamp)
    ]
    
    if len(next_shots) > 0:
        next_shot_timestamp = next_shots.iloc[0]['timestamp']
    else:
        next_shot_timestamp = None
    
    # Get subsequent frames in the same rally after the hit
    if next_shot_timestamp is not None:
        # Only look BEFORE the next shot
        after_hit = df[
            (df['rally_id'] == rally_id) & 
            (df['timestamp'] > hit_timestamp) &
            (df['timestamp'] < next_shot_timestamp)
        ]
    else:
        # No next shot, look until end of rally
        after_hit = df[
            (df['rally_id'] == rally_id) & 
            (df['timestamp'] > hit_timestamp)
        ]
    
    # Check which player hit and track their return to T
    if player_id == 1:
        in_t_col = 'p1_in_t_zone'
    else:
        in_t_col = 'p2_in_t_zone'
    
    # Find first frame where player re-enters T-zone
    if len(after_hit) > 0:
        returned_to_t = after_hit[after_hit[in_t_col] == True]
    else:
        returned_to_t = pd.DataFrame()  # Empty, no frames to check
    
    if len(returned_to_t) > 0:
        first_return = returned_to_t.iloc[0]
        time_to_t = first_return['timestamp'] - hit_timestamp
        
        time_to_t_results.append({
            'rally_id': rally_id,
            'player_id': player_id,
            'hit_timestamp': hit_timestamp,
            'return_timestamp': first_return['timestamp'],
            'time_to_t_seconds': time_to_t,
            'returned_before_next_shot': True
        })
    else:
        # Player did NOT return to T before next shot - record as failure
        time_to_t_results.append({
            'rally_id': rally_id,
            'player_id': player_id,
            'hit_timestamp': hit_timestamp,
            'return_timestamp': None,
            'time_to_t_seconds': None,
            'returned_before_next_shot': False
        })

# Convert to DataFrame
time_to_t_df = pd.DataFrame(time_to_t_results)

if len(time_to_t_df) > 0:
    # Filter to only successful returns
    successful_returns = time_to_t_df[time_to_t_df['returned_before_next_shot'] == True]
    
    if len(successful_returns) > 0:
        # Calculate average time-to-T per player
        p1_avg_time = successful_returns[successful_returns['player_id'] == 1]['time_to_t_seconds'].mean()
        p2_avg_time = successful_returns[successful_returns['player_id'] == 2]['time_to_t_seconds'].mean()
        
        # Calculate success rate (% of shots where player returned to T)
        p1_success_rate = (successful_returns['player_id'] == 1).sum() / (time_to_t_df['player_id'] == 1).sum() * 100
        p2_success_rate = (successful_returns['player_id'] == 2).sum() / (time_to_t_df['player_id'] == 2).sum() * 100
        
        print(f"Player 1 - Avg Time-to-T: {p1_avg_time:.2f} seconds (Success Rate: {p1_success_rate:.1f}%)")
        print(f"Player 2 - Avg Time-to-T: {p2_avg_time:.2f} seconds (Success Rate: {p2_success_rate:.1f}%)")
        
        # Visualization 1: Average Time-to-T Comparison
        fig1 = go.Figure()
        
        fig1.add_trace(go.Bar(
            x=['Player 1', 'Player 2'],
            y=[p1_avg_time, p2_avg_time],
            marker_color=['#636EFA', '#EF553B'],
            opacity=0.8,
            text=[f'{p1_avg_time:.2f}s', f'{p2_avg_time:.2f}s'],
            textposition='auto'
        ))
        
        fig1.update_layout(
            title='Average Time-to-T After Shot (Successful Returns Only)',
            xaxis_title='Player',
            yaxis_title='Time (seconds)',
            template='plotly_white',
            height=500,
            width=600,
            showlegend=False
        )
        
        fig1.show()
        
        # Visualization 2: Success Rate
        fig2 = go.Figure()
        
        fig2.add_trace(go.Bar(
            x=['Player 1', 'Player 2'],
            y=[p1_success_rate, p2_success_rate],
            marker_color=['#636EFA', '#EF553B'],
            opacity=0.8,
            text=[f'{p1_success_rate:.1f}%', f'{p2_success_rate:.1f}%'],
            textposition='auto'
        ))
        
        fig2.update_layout(
            title='T-Zone Return Success Rate (% of shots)',
            xaxis_title='Player',
            yaxis_title='Success Rate (%)',
            template='plotly_white',
            height=500,
            width=600,
            showlegend=False,
            yaxis=dict(range=[0, 100])
        )
        
        fig2.show()
        
        # Visualization 3: Distribution of Time-to-T
        fig3 = go.Figure()
        
        p1_times = successful_returns[successful_returns['player_id'] == 1]['time_to_t_seconds']
        p2_times = successful_returns[successful_returns['player_id'] == 2]['time_to_t_seconds']
        
        fig3.add_trace(go.Box(
            y=p1_times,
            name='Player 1',
            marker_color='#636EFA',
            boxmean='sd'
        ))
        
        fig3.add_trace(go.Box(
            y=p2_times,
            name='Player 2',
            marker_color='#EF553B',
            boxmean='sd'
        ))
        
        fig3.update_layout(
            title='Time-to-T Distribution',
            yaxis_title='Time (seconds)',
            template='plotly_white',
            height=500,
            width=700
        )
        
        fig3.show()
        
        print(f"\nTotal shots analyzed: {len(time_to_t_df)}")
        print(f"Player 1 - Successful returns: {len(p1_times)} / {(time_to_t_df['player_id'] == 1).sum()}")
        print(f"Player 2 - Successful returns: {len(p2_times)} / {(time_to_t_df['player_id'] == 2).sum()}")
    else:
        print("No successful returns to T found")
else:
    print("No time-to-T data available")

Player 1 - Avg Time-to-T: 1.13 seconds (Success Rate: 54.4%)
Player 2 - Avg Time-to-T: 1.32 seconds (Success Rate: 59.6%)



Total shots analyzed: 114
Player 1 - Successful returns: 31 / 57
Player 2 - Successful returns: 34 / 57


### **SQL Queries**

#### Total Shots Per player

In [46]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_total_shots_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  COUNT(*) AS total_shots
FROM t_returns
GROUP BY player_id
ORDER BY player_id
"""

total_shots_result = pysqldf(time_to_t_total_shots_query)
print("Total shots per player:")
print(total_shots_result)


Total shots per player:
   player_id  total_shots
0          1           57
1          2           57


#### Successful Returns Per Player

In [47]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_successful_returns_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  SUM(CASE WHEN return_timestamp IS NOT NULL THEN 1 ELSE 0 END) AS successful_returns
FROM t_returns
GROUP BY player_id
ORDER BY player_id
"""

successful_returns_result = pysqldf(time_to_t_successful_returns_query)
print("Successful returns per player:")
print(successful_returns_result)


Successful returns per player:
   player_id  successful_returns
0          1                  31
1          2                  34


#### Success Rate Per Player

In [48]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_success_rate_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  ROUND(100.0 * SUM(CASE WHEN return_timestamp IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate_percent
FROM t_returns
GROUP BY player_id
ORDER BY player_id
"""

success_rate_result = pysqldf(time_to_t_success_rate_query)
print("Success rate (%) per player:")
print(success_rate_result)


Success rate (%) per player:
   player_id  success_rate_percent
0          1                 54.39
1          2                 59.65


#### Average Time-to-T (Successful Returns Only)

In [49]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_avg_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  ROUND(AVG(return_timestamp - hit_timestamp), 2) AS avg_time_to_t_seconds
FROM t_returns
WHERE return_timestamp IS NOT NULL
GROUP BY player_id
ORDER BY player_id
"""

avg_time_result = pysqldf(time_to_t_avg_query)
print("Average Time-to-T (successful returns only):")
print(avg_time_result)


Average Time-to-T (successful returns only):
   player_id  avg_time_to_t_seconds
0          1                   1.13
1          2                   1.32


#### Min Time to T



In [56]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_min_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  ROUND(MIN(return_timestamp - hit_timestamp), 2) AS min_time_to_t_seconds
FROM t_returns
WHERE return_timestamp IS NOT NULL
GROUP BY player_id
ORDER BY player_id
"""

min_time_result = pysqldf(time_to_t_min_query)
print("Minimum Time-to-T (successful returns only):")
print(min_time_result)


Minimum Time-to-T (successful returns only):
   player_id  min_time_to_t_seconds
0          1                   0.02
1          2                   0.02


#### Max time to T

In [57]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_max_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  ROUND(MAX(return_timestamp - hit_timestamp), 2) AS max_time_to_t_seconds
FROM t_returns
WHERE return_timestamp IS NOT NULL
GROUP BY player_id
ORDER BY player_id
"""

max_time_result = pysqldf(time_to_t_max_query)
print("Maximum Time-to-T (successful returns only):")
print(max_time_result)


Maximum Time-to-T (successful returns only):
   player_id  max_time_to_t_seconds
0          1                   2.48
1          2                   2.83


#### Variance of time to T

In [58]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_to_t_variance_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp AS hit_timestamp,
    racket_hit_player_id,
    LEAD(timestamp) OVER (PARTITION BY rally_id, racket_hit_player_id ORDER BY timestamp) AS next_shot_timestamp
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
t_returns AS (
  SELECT 
    h.rally_id,
    h.racket_hit_player_id AS player_id,
    h.hit_timestamp,
    h.next_shot_timestamp,
    MIN(
      CASE 
        WHEN h.racket_hit_player_id = 1 AND d.p1_in_t_zone = 1 THEN d.timestamp
        WHEN h.racket_hit_player_id = 2 AND d.p2_in_t_zone = 1 THEN d.timestamp
      END
    ) AS return_timestamp
  FROM racket_hits h
  LEFT JOIN df d
    ON h.rally_id = d.rally_id
    AND d.timestamp > h.hit_timestamp
    AND (h.next_shot_timestamp IS NULL OR d.timestamp < h.next_shot_timestamp)
  GROUP BY h.rally_id, h.racket_hit_player_id, h.hit_timestamp, h.next_shot_timestamp
)
SELECT
  player_id,
  ROUND(
    AVG((return_timestamp - hit_timestamp) * (return_timestamp - hit_timestamp)) 
    - AVG(return_timestamp - hit_timestamp) * AVG(return_timestamp - hit_timestamp), 2
  ) AS variance_time_to_t
FROM t_returns
WHERE return_timestamp IS NOT NULL
GROUP BY player_id
ORDER BY player_id
"""

variance_time_result = pysqldf(time_to_t_variance_query)
print("Variance of Time-to-T (successful returns only):")
print(variance_time_result)


Variance of Time-to-T (successful returns only):
   player_id  variance_time_to_t
0          1                0.55
1          2                0.48


## **4. Time in T**

In [50]:
# Calculate overall % time in T-zone
total_frames = len(df_sorted)
p1_frames_in_t = df_sorted['p1_in_t_zone'].sum()
p2_frames_in_t = df_sorted['p2_in_t_zone'].sum()

p1_percent_in_t = (p1_frames_in_t / total_frames) * 100
p2_percent_in_t = (p2_frames_in_t / total_frames) * 100

print(f"Player 1 - % Time in T-zone: {p1_percent_in_t:.2f}%")
print(f"Player 2 - % Time in T-zone: {p2_percent_in_t:.2f}%")

# Calculate % time in T-zone per rally
rally_t_stats = df_sorted.groupby('rally_id').agg({
    'p1_in_t_zone': ['sum', 'count'],
    'p2_in_t_zone': ['sum', 'count']
}).reset_index()

rally_t_stats.columns = ['rally_id', 'p1_frames_in_t', 'total_frames_p1', 'p2_frames_in_t', 'total_frames_p2']
rally_t_stats['p1_percent_in_t'] = (rally_t_stats['p1_frames_in_t'] / rally_t_stats['total_frames_p1']) * 100
rally_t_stats['p2_percent_in_t'] = (rally_t_stats['p2_frames_in_t'] / rally_t_stats['total_frames_p2']) * 100

# Visualization 1: Overall % Time in T-zone (Bar Chart)
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=['Player 1', 'Player 2'],
    y=[p1_percent_in_t, p2_percent_in_t],
    marker_color=['#636EFA', '#EF553B'],
    opacity=0.8,
    text=[f'{p1_percent_in_t:.1f}%', f'{p2_percent_in_t:.1f}%'],
    textposition='auto'
))

fig1.update_layout(
    title='Overall % of Time Spent in T-Zone',
    xaxis_title='Player',
    yaxis_title='% Time in T-Zone',
    template='plotly_white',
    height=500,
    width=600,
    showlegend=False,
    yaxis=dict(range=[0, 100])
)

fig1.show()

# Visualization 2: % Time in T-zone Per Rally (Line Chart)
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=rally_t_stats['rally_id'],
    y=rally_t_stats['p1_percent_in_t'],
    mode='lines+markers',
    name='Player 1',
    line=dict(color='#636EFA', width=2),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=rally_t_stats['rally_id'],
    y=rally_t_stats['p2_percent_in_t'],
    mode='lines+markers',
    name='Player 2',
    line=dict(color='#EF553B', width=2),
    marker=dict(size=6)
))

fig2.update_layout(
    title='% Time in T-Zone Per Rally',
    xaxis_title='Rally ID',
    yaxis_title='% Time in T-Zone',
    template='plotly_white',
    height=500,
    width=1000,
    yaxis=dict(range=[0, 100]),
    hovermode='x unified'
)

fig2.show()

# Summary statistics
print(f"\nPlayer 1 - Avg % in T per rally: {rally_t_stats['p1_percent_in_t'].mean():.2f}%")
print(f"Player 2 - Avg % in T per rally: {rally_t_stats['p2_percent_in_t'].mean():.2f}%")
print(f"\nPlayer 1 - Best rally: {rally_t_stats['p1_percent_in_t'].max():.2f}%")
print(f"Player 2 - Best rally: {rally_t_stats['p2_percent_in_t'].max():.2f}%")

Player 1 - % Time in T-zone: 15.16%
Player 2 - % Time in T-zone: 14.22%



Player 1 - Avg % in T per rally: 14.73%
Player 2 - Avg % in T per rally: 13.81%

Player 1 - Best rally: 36.72%
Player 2 - Best rally: 24.96%


### **SQL Queries**

#### Overall % of Time in T per Player

In [64]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""

time_in_t_overall_query = f"""
WITH t_zone_data AS (
    SELECT
        rally_id,
        timestamp,
        p1_in_t_zone,
        p2_in_t_zone
    FROM df
    WHERE 1=1 {filter_clause}
)
SELECT
    1 AS player_id,
    ROUND(100.0 * SUM(p1_in_t_zone) / COUNT(*), 2) AS percent_in_t
FROM t_zone_data
WHERE 1=1 {"AND 1=1" if player_id is None else f"AND 1=1"} -- placeholder
UNION ALL
SELECT
    2 AS player_id,
    ROUND(100.0 * SUM(p2_in_t_zone) / COUNT(*), 2) AS percent_in_t
FROM t_zone_data
WHERE 1=1 {"AND 1=1" if player_id is None else f"AND 1=1"} -- placeholder
ORDER BY player_id
"""

overall_result = pysqldf(time_in_t_overall_query)

# Apply player filter in Python
if player_id is not None:
    overall_result = overall_result[overall_result['player_id'] == player_id]

print("Overall % of Time-in-T per player:")
print(overall_result)


Overall % of Time-in-T per player:
   player_id  percent_in_t
0          1         15.16
1          2         14.22


#### Per-rally % Time-in-T per player

In [69]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
per_rally_time_in_t_query = f"""
WITH t_zone_data AS (
    SELECT
        rally_id,
        timestamp,
        p1_in_t_zone,
        p2_in_t_zone
    FROM df
    WHERE 1=1 {filter_clause}
)
SELECT
    rally_id,
    ROUND(100.0 * SUM(p1_in_t_zone) / COUNT(*), 2) AS p1_percent_in_t,
    ROUND(100.0 * SUM(p2_in_t_zone) / COUNT(*), 2) AS p2_percent_in_t
FROM t_zone_data
GROUP BY rally_id
ORDER BY rally_id
"""

per_rally_result = pysqldf(per_rally_time_in_t_query)
print("Per-rally % Time-in-T per player:")
print(per_rally_result)



Per-rally % Time-in-T per player:
    rally_id  p1_percent_in_t  p2_percent_in_t
0          0             8.40            14.44
1          1            17.74            23.87
2          2             0.00            14.96
3          3            14.85            21.78
4          4            36.72            24.96
5          5             6.39             7.96
6          6            15.92             6.41
7          7            25.51            20.81
8          8            13.65             3.36
9          9            11.57             0.00
10        10            11.29            13.35


#### Average % Time-in-T per rally (aggregated)

In [70]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
avg_percent_in_t_query = f"""
WITH t_zone_data AS (
    SELECT
        rally_id,
        timestamp,
        p1_in_t_zone,
        p2_in_t_zone
    FROM df
    WHERE 1=1 {filter_clause}
),
per_rally_stats AS (
    SELECT
        rally_id,
        ROUND(100.0 * SUM(p1_in_t_zone) / COUNT(*), 2) AS p1_percent_in_t,
        ROUND(100.0 * SUM(p2_in_t_zone) / COUNT(*), 2) AS p2_percent_in_t
    FROM t_zone_data
    GROUP BY rally_id
)
SELECT
    ROUND(AVG(p1_percent_in_t), 2) AS p1_avg_percent_in_t,
    ROUND(AVG(p2_percent_in_t), 2) AS p2_avg_percent_in_t
FROM per_rally_stats
"""

avg_percent_result = pysqldf(avg_percent_in_t_query)
print("Average % Time-in-T per rally:")
print(avg_percent_result)


Average % Time-in-T per rally:
   p1_avg_percent_in_t  p2_avg_percent_in_t
0                14.73                13.81


#### Best rally % Time-in-T per player

In [73]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
best_rally_percent_query = f"""
WITH t_zone_data AS (
    SELECT
        rally_id,
        timestamp,
        p1_in_t_zone,
        p2_in_t_zone
    FROM df
    WHERE 1=1 {filter_clause}
),
per_rally_stats AS (
    SELECT
        rally_id,
        ROUND(100.0 * SUM(p1_in_t_zone) / COUNT(*), 2) AS p1_percent_in_t,
        ROUND(100.0 * SUM(p2_in_t_zone) / COUNT(*), 2) AS p2_percent_in_t
    FROM t_zone_data
    GROUP BY rally_id
)
SELECT
    (SELECT p1_percent_in_t FROM per_rally_stats ORDER BY p1_percent_in_t DESC LIMIT 1) AS p1_best_rally_percent,
    (SELECT p2_percent_in_t FROM per_rally_stats ORDER BY p2_percent_in_t DESC LIMIT 1) AS p2_best_rally_percent
"""

best_rally_result = pysqldf(best_rally_percent_query)
print("Best rally % Time-in-T per player:")
print(best_rally_result)


Best rally % Time-in-T per player:
   p1_best_rally_percent  p2_best_rally_percent
0                  36.72                  24.96


## **5. Shot Displacement From T**

In [52]:
# Opponent Displacement from T - Rewritten for clarity

import plotly.graph_objects as go
import pandas as pd

# Get all racket hit events sorted by time
racket_hits = df[df['is_racket_hit'] == True].sort_values(['rally_id', 'timestamp']).copy()

# Calculate displacement for each shot
displacement_results = []

for idx, hit in racket_hits.iterrows():
    shooter_id = hit['racket_hit_player_id']
    target_id = 2 if shooter_id == 1 else 1  # The player being displaced
    rally_id = hit['rally_id']
    hit_timestamp = hit['timestamp']
    
    # Get target player's position at the shooter's hit
    if target_id == 1:
        target_distance_at_shot = hit['p1_distance_from_t']
    else:
        target_distance_at_shot = hit['p2_distance_from_t']
    
    # Find target player's NEXT shot (their response)
    target_next_shots = racket_hits[
        (racket_hits['rally_id'] == rally_id) &
        (racket_hits['racket_hit_player_id'] == target_id) &
        (racket_hits['timestamp'] > hit_timestamp)
    ]
    
    if len(target_next_shots) > 0:
        target_response = target_next_shots.iloc[0]
        
        # Get target player's position at their response hit
        if target_id == 1:
            target_distance_at_response = target_response['p1_distance_from_t']
        else:
            target_distance_at_response = target_response['p2_distance_from_t']
        
        # Calculate displacement (change in distance from T)
        displacement = target_distance_at_response - target_distance_at_shot
        
        displacement_results.append({
            'rally_id': rally_id,
            'shooter_id': shooter_id,  # Who hit the shot
            'displaced_player_id': target_id,  # Who got displaced
            'hit_timestamp': hit_timestamp,
            'response_timestamp': target_response['timestamp'],
            'distance_at_shot': target_distance_at_shot,
            'distance_at_response': target_distance_at_response,
            'displacement_from_t': displacement  # Positive = moved away from T
        })

# Convert to DataFrame
displacement_df = pd.DataFrame(displacement_results)

if len(displacement_df) > 0:
    # Group by displaced player (who got moved)
    p1_displaced = displacement_df[displacement_df['displaced_player_id'] == 1]  # Player 1 was displaced by Player 2
    p2_displaced = displacement_df[displacement_df['displaced_player_id'] == 2]  # Player 2 was displaced by Player 1
    
    print("=== Opponent Displacement from T ===\n")
    
    if len(p1_displaced) > 0:
        print(f"Player 1 Displacement (displaced by Player 2's shots):")
        print(f"  Average displacement: {p1_displaced['displacement_from_t'].mean():.2f}m")
        print(f"  Max displacement: {p1_displaced['displacement_from_t'].max():.2f}m")
        print(f"  Min displacement: {p1_displaced['displacement_from_t'].min():.2f}m")
        print(f"  Total shots received: {len(p1_displaced)}\n")
    
    if len(p2_displaced) > 0:
        print(f"Player 2 Displacement (displaced by Player 1's shots):")
        print(f"  Average displacement: {p2_displaced['displacement_from_t'].mean():.2f}m")
        print(f"  Max displacement: {p2_displaced['displacement_from_t'].max():.2f}m")
        print(f"  Min displacement: {p2_displaced['displacement_from_t'].min():.2f}m")
        print(f"  Total shots received: {len(p2_displaced)}\n")
    
    # Visualization 1: Average Displacement
    fig1 = go.Figure()
    
    fig1.add_trace(go.Bar(
        x=['Player 1 Displacement', 'Player 2 Displacement'],
        y=[
            p1_displaced['displacement_from_t'].mean(),
            p2_displaced['displacement_from_t'].mean()
        ],
        marker_color=['#636EFA', '#EF553B'],
        opacity=0.8,
        text=[
            f"{p1_displaced['displacement_from_t'].mean():.2f}m",
            f"{p2_displaced['displacement_from_t'].mean():.2f}m"
        ],
        textposition='auto'
    ))
    
    fig1.update_layout(
        title='Average Displacement from T (How much each player was moved)',
        xaxis_title='Player',
        yaxis_title='Displacement (meters)',
        template='plotly_white',
        height=500,
        width=600,
        showlegend=False
    )
    
    fig1.show()
    
    # Visualization 2: Distribution of Displacement
    fig2 = go.Figure()
    
    fig2.add_trace(go.Box(
        y=p1_displaced['displacement_from_t'],
        name='Player 1 Displacement',
        marker_color='#636EFA',
        boxmean='sd'
    ))
    
    fig2.add_trace(go.Box(
        y=p2_displaced['displacement_from_t'],
        name='Player 2 Displacement',
        marker_color='#EF553B',
        boxmean='sd'
    ))
    
    fig2.update_layout(
        title='Distribution of Displacement from T',
        yaxis_title='Displacement (meters)',
        template='plotly_white',
        height=500,
        width=700
    )
    
    fig2.show()
    
    # Visualization 3: Displacement Over Time (Line Graph)
    fig3 = go.Figure()
    
    p1_sorted = p1_displaced.sort_values('hit_timestamp')
    p2_sorted = p2_displaced.sort_values('hit_timestamp')
    
    fig3.add_trace(go.Scatter(
        x=p1_sorted['hit_timestamp'],
        y=p1_sorted['displacement_from_t'],
        mode='lines+markers',
        name='Player 1 Displacement',
        line=dict(color='#636EFA', width=2),
        marker=dict(size=6)
    ))
    
    fig3.add_trace(go.Scatter(
        x=p2_sorted['hit_timestamp'],
        y=p2_sorted['displacement_from_t'],
        mode='lines+markers',
        name='Player 2 Displacement',
        line=dict(color='#EF553B', width=2),
        marker=dict(size=6)
    ))
    
    fig3.add_hline(y=0, line_dash="dash", line_color="gray", 
                   annotation_text="No displacement")
    
    fig3.update_layout(
        title='Displacement Over Time',
        xaxis_title='Time (seconds)',
        yaxis_title='Displacement (meters)',
        template='plotly_white',
        height=500,
        width=1000,
        hovermode='x unified'
    )
    
    fig3.show()
    
    # Visualization 4: Displacement Per Rally (Line Graph)
    fig4 = go.Figure()
    
    fig4.add_trace(go.Scatter(
        x=p1_displaced['rally_id'],
        y=p1_displaced['displacement_from_t'],
        mode='lines+markers',
        name='Player 1 Displacement',
        line=dict(color='#636EFA', width=2),
        marker=dict(size=6)
    ))
    
    fig4.add_trace(go.Scatter(
        x=p2_displaced['rally_id'],
        y=p2_displaced['displacement_from_t'],
        mode='lines+markers',
        name='Player 2 Displacement',
        line=dict(color='#EF553B', width=2),
        marker=dict(size=6)
    ))
    
    fig4.add_hline(y=0, line_dash="dash", line_color="gray", 
                   annotation_text="No displacement")
    
    fig4.update_layout(
        title='Displacement Per Rally',
        xaxis_title='Rally ID',
        yaxis_title='Displacement (meters)',
        template='plotly_white',
        height=500,
        width=1000,
        hovermode='x unified'
    )
    
    fig4.show()
    
else:
    print("No displacement data available")

=== Opponent Displacement from T ===

Player 1 Displacement (displaced by Player 2's shots):
  Average displacement: 1.00m
  Max displacement: 3.02m
  Min displacement: -0.73m
  Total shots received: 52

Player 2 Displacement (displaced by Player 1's shots):
  Average displacement: 1.04m
  Max displacement: 2.85m
  Min displacement: -0.35m
  Total shots received: 51



### **SQL Queries**

#### Average Displacement

In [74]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
avg_displacement_query = f"""
WITH racket_hits AS (
  SELECT 
    rally_id,
    timestamp,
    racket_hit_player_id AS shooter_id,
    p1_distance_from_t,
    p2_distance_from_t,
    CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS displaced_player_id
  FROM df
  WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
  SELECT 
    h1.rally_id,
    h1.shooter_id,
    h1.displaced_player_id,
    h1.timestamp AS hit_timestamp,
    h1.p1_distance_from_t AS h1_p1_dist,
    h1.p2_distance_from_t AS h1_p2_dist,
    h2.timestamp AS response_timestamp,
    h2.p1_distance_from_t AS h2_p1_dist,
    h2.p2_distance_from_t AS h2_p2_dist
  FROM racket_hits h1
  LEFT JOIN racket_hits h2
    ON h1.rally_id = h2.rally_id
    AND h2.shooter_id = h1.displaced_player_id
    AND h2.timestamp > h1.timestamp
    AND h2.timestamp = (
      SELECT MIN(h3.timestamp)
      FROM racket_hits h3
      WHERE h3.rally_id = h1.rally_id
        AND h3.shooter_id = h1.displaced_player_id
        AND h3.timestamp > h1.timestamp
    )
  WHERE h2.timestamp IS NOT NULL
),
displacement_calc AS (
  SELECT
    displaced_player_id,
    CASE WHEN displaced_player_id = 1 THEN ROUND(h2_p1_dist - h1_p1_dist, 2)
         ELSE ROUND(h2_p2_dist - h1_p2_dist, 2)
    END AS displacement_from_t
  FROM shots_with_responses
)
SELECT displaced_player_id AS player_id,
       ROUND(AVG(displacement_from_t), 2) AS avg_displacement
FROM displacement_calc
GROUP BY displaced_player_id
ORDER BY player_id
"""

avg_displacement_result = pysqldf(avg_displacement_query)
print("Average Displacement per player:")
print(avg_displacement_result)


Average Displacement per player:
   player_id  avg_displacement
0          1              1.00
1          2              1.04


In [75]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
min_displacement_query = f"""
WITH racket_hits AS (
    SELECT 
        rally_id,
        timestamp,
        racket_hit_player_id AS shooter_id,
        p1_distance_from_t,
        p2_distance_from_t,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS displaced_player_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT 
        h1.rally_id,
        h1.shooter_id,
        h1.displaced_player_id,
        h1.timestamp AS hit_timestamp,
        h1.p1_distance_from_t AS h1_p1_dist,
        h1.p2_distance_from_t AS h1_p2_dist,
        h2.timestamp AS response_timestamp,
        h2.p1_distance_from_t AS h2_p1_dist,
        h2.p2_distance_from_t AS h2_p2_dist
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.shooter_id = h1.displaced_player_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
            AND h3.shooter_id = h1.displaced_player_id
            AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
),
displacement_calc AS (
    SELECT
        displaced_player_id,
        CASE WHEN displaced_player_id = 1 THEN ROUND(h2_p1_dist - h1_p1_dist, 2)
             ELSE ROUND(h2_p2_dist - h1_p2_dist, 2)
        END AS displacement_from_t
    FROM shots_with_responses
)
SELECT displaced_player_id AS player_id,
       ROUND(MIN(displacement_from_t), 2) AS min_displacement
FROM displacement_calc
GROUP BY displaced_player_id
ORDER BY player_id
"""

min_displacement_result = pysqldf(min_displacement_query)
print("Minimum Displacement per player:")
print(min_displacement_result)


Minimum Displacement per player:
   player_id  min_displacement
0          1             -0.73
1          2             -0.35


#### Maximum Displacement per player

In [76]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
max_displacement_query = f"""

WITH racket_hits AS (
    SELECT 
        rally_id,
        timestamp,
        racket_hit_player_id AS shooter_id,
        p1_distance_from_t,
        p2_distance_from_t,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS displaced_player_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT 
        h1.rally_id,
        h1.shooter_id,
        h1.displaced_player_id,
        h1.timestamp AS hit_timestamp,
        h1.p1_distance_from_t AS h1_p1_dist,
        h1.p2_distance_from_t AS h1_p2_dist,
        h2.timestamp AS response_timestamp,
        h2.p1_distance_from_t AS h2_p1_dist,
        h2.p2_distance_from_t AS h2_p2_dist
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.shooter_id = h1.displaced_player_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
            AND h3.shooter_id = h1.displaced_player_id
            AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
),
displacement_calc AS (
    SELECT
        displaced_player_id,
        CASE WHEN displaced_player_id = 1 THEN ROUND(h2_p1_dist - h1_p1_dist, 2)
             ELSE ROUND(h2_p2_dist - h1_p2_dist, 2)
        END AS displacement_from_t
    FROM shots_with_responses
)
SELECT displaced_player_id AS player_id,
       ROUND(MAX(displacement_from_t), 2) AS max_displacement
FROM displacement_calc
GROUP BY displaced_player_id
ORDER BY player_id
"""

max_displacement_result = pysqldf(max_displacement_query)
print("Maximum Displacement per player:")
print(max_displacement_result)


Maximum Displacement per player:
   player_id  max_displacement
0          1              3.02
1          2              2.85


#### Variance of Displacement per player

In [77]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
variance_displacement_query = f"""
WITH racket_hits AS (
    SELECT 
        rally_id,
        timestamp,
        racket_hit_player_id AS shooter_id,
        p1_distance_from_t,
        p2_distance_from_t,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS displaced_player_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT 
        h1.rally_id,
        h1.shooter_id,
        h1.displaced_player_id,
        h1.timestamp AS hit_timestamp,
        h1.p1_distance_from_t AS h1_p1_dist,
        h1.p2_distance_from_t AS h1_p2_dist,
        h2.timestamp AS response_timestamp,
        h2.p1_distance_from_t AS h2_p1_dist,
        h2.p2_distance_from_t AS h2_p2_dist
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.shooter_id = h1.displaced_player_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
            AND h3.shooter_id = h1.displaced_player_id
            AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
),
displacement_calc AS (
    SELECT
        displaced_player_id,
        CASE WHEN displaced_player_id = 1 THEN ROUND(h2_p1_dist - h1_p1_dist, 2)
             ELSE ROUND(h2_p2_dist - h1_p2_dist, 2)
        END AS displacement_from_t
    FROM shots_with_responses
),
stats AS (
    SELECT displaced_player_id AS player_id,
           AVG(displacement_from_t) AS mean,
           AVG(displacement_from_t * displacement_from_t) AS mean_sq
    FROM displacement_calc
    GROUP BY displaced_player_id
)
SELECT player_id,
       ROUND(mean_sq - mean * mean, 2) AS variance_displacement
FROM stats
ORDER BY player_id
"""

variance_displacement_result = pysqldf(variance_displacement_query)
print("Variance of Displacement per player:")
print(variance_displacement_result)


Variance of Displacement per player:
   player_id  variance_displacement
0          1                   1.09
1          2                   0.74


#### Shot Details (for visualizations)

In [78]:
rally_id = None
player_id = None
start_time = None
end_time = None

conditions = []

if rally_id is not None:
    conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(conditions) if conditions else ""
shot_detail_query = f"""
WITH racket_hits AS (
    SELECT 
        rally_id,
        timestamp,
        racket_hit_player_id AS shooter_id,
        p1_distance_from_t,
        p2_distance_from_t,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS displaced_player_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT 
        h1.rally_id,
        h1.shooter_id,
        h1.displaced_player_id,
        h1.timestamp AS hit_timestamp,
        h1.p1_distance_from_t AS h1_p1_dist,
        h1.p2_distance_from_t AS h1_p2_dist,
        h2.timestamp AS response_timestamp,
        h2.p1_distance_from_t AS h2_p1_dist,
        h2.p2_distance_from_t AS h2_p2_dist
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.shooter_id = h1.displaced_player_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
            AND h3.shooter_id = h1.displaced_player_id
            AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT displaced_player_id AS player_id,
       rally_id,
       hit_timestamp,
       CASE WHEN displaced_player_id = 1 THEN ROUND(h2_p1_dist - h1_p1_dist, 2)
            ELSE ROUND(h2_p2_dist - h1_p2_dist, 2)
       END AS displacement_from_t,
       CASE WHEN displaced_player_id = 1 THEN h1_p1_dist ELSE h1_p2_dist END AS distance_at_shot,
       CASE WHEN displaced_player_id = 1 THEN h2_p1_dist ELSE h2_p2_dist END AS distance_at_response
FROM shots_with_responses
ORDER BY player_id, rally_id, hit_timestamp
"""

shot_detail_result = pysqldf(shot_detail_query)
print("Shot-level Displacement Details:")
print(shot_detail_result)


Shot-level Displacement Details:
     player_id  rally_id  hit_timestamp  displacement_from_t  \
0            1         0       1.466667                 0.89   
1            1         0       4.650000                 2.88   
2            1         0       7.683333                -0.53   
3            1         0      11.183333                -0.49   
4            1         0      12.133333                -0.46   
..         ...       ...            ...                  ...   
98           2         9     266.883333                 2.23   
99           2         9     269.833333                 0.97   
100          2        10     287.616667                 0.61   
101          2        10     292.800000                 2.82   
102          2        10     296.016667                 1.34   

     distance_at_shot  distance_at_response  
0            2.614171              3.504224  
1            1.289616              4.171103  
2            3.050771              2.519227  
3            3

## **6. Depth Dominance**

In [84]:
# Depth Dominance - Using Player Positions in Meters (Correct Column Names)

import plotly.graph_objects as go
import pandas as pd

# Get all racket hit events sorted by time
racket_hits = df[df['is_racket_hit'] == True].sort_values(['rally_id', 'timestamp']).copy()

# Calculate depth dominance for each shot using player positions (meters)
depth_results = []

for idx, hit in racket_hits.iterrows():
    player_id = hit['racket_hit_player_id']
    opponent_id = 2 if player_id == 1 else 1
    rally_id = hit['rally_id']
    hit_timestamp = hit['timestamp']
    
    # Get player's Y position at this hit
    my_shot_depth = hit['player_1_y_meter'] if player_id == 1 else hit['player_2_y_meter']
    
    # Find opponent's NEXT shot (their response)
    opponent_next_shots = racket_hits[
        (racket_hits['rally_id'] == rally_id) &
        (racket_hits['racket_hit_player_id'] == opponent_id) &
        (racket_hits['timestamp'] > hit_timestamp)
    ]
    
    if len(opponent_next_shots) > 0:
        opponent_response = opponent_next_shots.iloc[0]
        
        # Get opponent's Y position at their response
        opponent_shot_depth = opponent_response['player_1_y_meter'] if opponent_id == 1 else opponent_response['player_2_y_meter']
        
        # Depth difference in meters (positive = opponent deeper)
        depth_difference = opponent_shot_depth - my_shot_depth
        opponent_is_deeper = depth_difference > 0
        
        depth_results.append({
            'rally_id': rally_id,
            'player_id': player_id,
            'opponent_id': opponent_id,
            'hit_timestamp': hit_timestamp,
            'opponent_shot_timestamp': opponent_response['timestamp'],
            'my_shot_depth_m': my_shot_depth,
            'opponent_shot_depth_m': opponent_shot_depth,
            'depth_difference_m': depth_difference,
            'opponent_is_deeper': opponent_is_deeper
        })

# Convert to DataFrame
depth_df = pd.DataFrame(depth_results)

if len(depth_df) > 0:
    # Split by player
    p1_depth = depth_df[depth_df['player_id'] == 1]
    p2_depth = depth_df[depth_df['player_id'] == 2]
    
    print("=== Depth Dominance (Player Positions in Meters) ===\n")
    
    if len(p1_depth) > 0:
        p1_dominance = (p1_depth['opponent_is_deeper'].sum() / len(p1_depth)) * 100
        print(f"Player 1's shots:")
        print(f"  Depth dominance: {p1_dominance:.1f}% ({p1_depth['opponent_is_deeper'].sum()}/{len(p1_depth)} shots)")
        print(f"  Average depth difference: {p1_depth['depth_difference_m'].mean():.2f}m")
        print(f"  Max depth advantage: {p1_depth['depth_difference_m'].max():.2f}m")
        print(f"  Min depth advantage: {p1_depth['depth_difference_m'].min():.2f}m")
        print(f"  Total shots: {len(p1_depth)}\n")
    
    if len(p2_depth) > 0:
        p2_dominance = (p2_depth['opponent_is_deeper'].sum() / len(p2_depth)) * 100
        print(f"Player 2's shots:")
        print(f"  Depth dominance: {p2_dominance:.1f}% ({p2_depth['opponent_is_deeper'].sum()}/{len(p2_depth)} shots)")
        print(f"  Average depth difference: {p2_depth['depth_difference_m'].mean():.2f}m")
        print(f"  Max depth advantage: {p2_depth['depth_difference_m'].max():.2f}m")
        print(f"  Min depth advantage: {p2_depth['depth_difference_m'].min():.2f}m")
        print(f"  Total shots: {len(p2_depth)}\n")
    
    # Visualization 1: Depth Dominance Percentage
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=['Player 1 Shots', 'Player 2 Shots'],
        y=[p1_dominance, p2_dominance],
        marker_color=['#636EFA', '#EF553B'],
        text=[f"{p1_dominance:.1f}%", f"{p2_dominance:.1f}%"],
        textposition='auto'
    ))
    fig1.update_layout(title='Depth Dominance (% of shots forcing opponent deeper)',
                       xaxis_title='Player', yaxis_title='Depth Dominance (%)',
                       template='plotly_white', height=500, width=600,
                       showlegend=False, yaxis=dict(range=[0,100]))
    fig1.show()
    
    # Visualization 2: Average Depth Difference
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=['Player 1 Shots', 'Player 2 Shots'],
        y=[p1_depth['depth_difference_m'].mean(), p2_depth['depth_difference_m'].mean()],
        text=[f"{p1_depth['depth_difference_m'].mean():.2f}m",
              f"{p2_depth['depth_difference_m'].mean():.2f}m"],
        textposition='auto',
        marker_color=['#636EFA', '#EF553B']
    ))
    fig2.update_layout(title='Average Depth Difference (Positive = Opponent Deeper)',
                       xaxis_title='Player', yaxis_title='Depth Difference (meters)',
                       template='plotly_white', height=500, width=600,
                       showlegend=False)
    fig2.show()
    
    # Visualization 3: Distribution of Depth Difference
    fig3 = go.Figure()
    fig3.add_trace(go.Box(y=p1_depth['depth_difference_m'], name='Player 1 Shots', marker_color='#636EFA', boxmean='sd'))
    fig3.add_trace(go.Box(y=p2_depth['depth_difference_m'], name='Player 2 Shots', marker_color='#EF553B', boxmean='sd'))
    fig3.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Equal depth")
    fig3.update_layout(title='Distribution of Depth Difference',
                       yaxis_title='Depth Difference (meters)',
                       template='plotly_white', height=500, width=700)
    fig3.show()
    
else:
    print("No depth data available")


=== Depth Dominance (Player Positions in Meters) ===

Player 1's shots:
  Depth dominance: 43.1% (22/51 shots)
  Average depth difference: -0.24m
  Max depth advantage: 4.18m
  Min depth advantage: -6.13m
  Total shots: 51

Player 2's shots:
  Depth dominance: 55.8% (29/52 shots)
  Average depth difference: 0.17m
  Max depth advantage: 5.34m
  Min depth advantage: -4.85m
  Total shots: 52



### **SQL Queries**

#### Depth Dominance Percentage (% shots opponent deeper)

In [87]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

# Query: Depth Dominance Percentage
depth_percentage_query = f"""
WITH racket_hits AS (
    SELECT
        rally_id,
        timestamp,
        racket_hit_player_id AS player_id,
        player_1_y_meter,
        player_2_y_meter,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT
        h1.player_id,
        CASE WHEN h2.player_id IS NOT NULL AND
                  ((CASE WHEN h2.player_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END) >
                   (CASE WHEN h1.player_id = 1 THEN h1.player_1_y_meter ELSE h1.player_2_y_meter END))
             THEN 1 ELSE 0 END AS opponent_is_deeper
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT
    player_id,
    ROUND(100.0 * SUM(opponent_is_deeper) / COUNT(*), 2) AS depth_dominance_percent
FROM shots_with_responses
GROUP BY player_id
ORDER BY player_id
"""

depth_percentage_result = pysqldf(depth_percentage_query)
print("Depth Dominance Percentage per Player:")
print(depth_percentage_result)


Depth Dominance Percentage per Player:
   player_id  depth_dominance_percent
0          1                    43.14
1          2                    55.77


#### Average Depth Difference

In [88]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

# Query: Average Depth Difference
avg_depth_query = f"""
WITH racket_hits AS (
    SELECT
        rally_id,
        timestamp,
        racket_hit_player_id AS player_id,
        player_1_y_meter,
        player_2_y_meter,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT
        h1.player_id,
        (CASE WHEN h2.player_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END) -
        (CASE WHEN h1.player_id = 1 THEN h1.player_1_y_meter ELSE h1.player_2_y_meter END) AS depth_difference_m
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT
    player_id,
    ROUND(AVG(depth_difference_m), 2) AS avg_depth_difference_m
FROM shots_with_responses
GROUP BY player_id
ORDER BY player_id
"""

avg_depth_result = pysqldf(avg_depth_query)
print("Average Depth Difference per Player (meters):")
print(avg_depth_result)


Average Depth Difference per Player (meters):
   player_id  avg_depth_difference_m
0          1                   -0.24
1          2                    0.17


#### Minimum Depth Difference

In [89]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

# Query: Minimum Depth Difference
min_depth_query = f"""
WITH racket_hits AS (
    SELECT
        rally_id,
        timestamp,
        racket_hit_player_id AS player_id,
        player_1_y_meter,
        player_2_y_meter,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT
        h1.player_id,
        (CASE WHEN h2.player_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END) -
        (CASE WHEN h1.player_id = 1 THEN h1.player_1_y_meter ELSE h1.player_2_y_meter END) AS depth_difference_m
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT
    player_id,
    ROUND(MIN(depth_difference_m), 2) AS min_depth_difference_m
FROM shots_with_responses
GROUP BY player_id
ORDER BY player_id
"""

min_depth_result = pysqldf(min_depth_query)
print("Minimum Depth Difference per Player (meters):")
print(min_depth_result)


Minimum Depth Difference per Player (meters):
   player_id  min_depth_difference_m
0          1                   -6.13
1          2                   -4.85


#### Maximum Depth Difference

In [90]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

# Query: Maximum Depth Difference
max_depth_query = f"""
WITH racket_hits AS (
    SELECT
        rally_id,
        timestamp,
        racket_hit_player_id AS player_id,
        player_1_y_meter,
        player_2_y_meter,
        CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
shots_with_responses AS (
    SELECT
        h1.player_id,
        (CASE WHEN h2.player_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END) -
        (CASE WHEN h1.player_id = 1 THEN h1.player_1_y_meter ELSE h1.player_2_y_meter END) AS depth_difference_m
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp > h1.timestamp
        AND h2.timestamp = (
            SELECT MIN(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp > h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT
    player_id,
    ROUND(MAX(depth_difference_m), 2) AS max_depth_difference_m
FROM shots_with_responses
GROUP BY player_id
ORDER BY player_id
"""

max_depth_result = pysqldf(max_depth_query)
print("Maximum Depth Difference per Player (meters):")
print(max_depth_result)


Maximum Depth Difference per Player (meters):
   player_id  max_depth_difference_m
0          1                    4.18
1          2                    5.34


## **7. Distance Travelled To Ball**

In [95]:
rally_id = None
player_id = None
start_time = None
end_time = None

# Get all racket hits sorted by rally and time
racket_hits = df[df['is_racket_hit'] == True].sort_values(['rally_id', 'timestamp']).copy()

# Apply filters if needed
if rally_id is not None:
    racket_hits = racket_hits[racket_hits['rally_id'] == rally_id]
if player_id is not None:
    racket_hits = racket_hits[racket_hits['racket_hit_player_id'] == player_id]
if start_time is not None:
    racket_hits = racket_hits[racket_hits['timestamp'] >= start_time]
if end_time is not None:
    racket_hits = racket_hits[racket_hits['timestamp'] <= end_time]

distance_results = []

for idx, hit in racket_hits.iterrows():
    player_id = hit['racket_hit_player_id']
    opponent_id = 2 if player_id == 1 else 1
    rally_id = hit['rally_id']
    hit_timestamp = hit['timestamp']
    
    # Player position at current hit
    my_x = hit['player_1_x_meter'] if player_id == 1 else hit['player_2_x_meter']
    my_y = hit['player_1_y_meter'] if player_id == 1 else hit['player_2_y_meter']
    
    # Find opponent's previous hit in the same rally
    opponent_prev_shots = racket_hits[
        (racket_hits['rally_id'] == rally_id) & 
        (racket_hits['racket_hit_player_id'] == opponent_id) &
        (racket_hits['timestamp'] < hit_timestamp)
    ]
    
    if len(opponent_prev_shots) > 0:
        opponent_last = opponent_prev_shots.iloc[-1]
        opp_x = opponent_last['player_1_x_meter'] if opponent_id == 1 else opponent_last['player_2_x_meter']
        opp_y = opponent_last['player_1_y_meter'] if opponent_id == 1 else opponent_last['player_2_y_meter']
        
        # Distance traveled to reach the ball
        distance = np.sqrt((my_x - opp_x)**2 + (my_y - opp_y)**2)
        
        distance_results.append({
            'rally_id': rally_id,
            'player_id': player_id,
            'opponent_id': opponent_id,
            'hit_timestamp': hit_timestamp,
            'distance_to_ball_m': distance
        })

distance_df = pd.DataFrame(distance_results)

# Summary per player
if len(distance_df) > 0:
    for pid in [1,2]:
        player_dist = distance_df[distance_df['player_id'] == pid]
        if len(player_dist) > 0:
            print(f"Player {pid} distance traveled stats:")
            print(f"  Total distance: {player_dist['distance_to_ball_m'].sum():.2f}m")
            print(f"  Average distance: {player_dist['distance_to_ball_m'].mean():.2f}m")
            print(f"  Max distance: {player_dist['distance_to_ball_m'].max():.2f}m")
            print(f"  Min distance: {player_dist['distance_to_ball_m'].min():.2f}m\n")
    
    # ----------------------
    # Line Graph: Distance over time
    # ----------------------
    fig = go.Figure()
    
    for pid in [1, 2]:
        player_dist = distance_df[distance_df['player_id'] == pid]
        if len(player_dist) > 0:
            fig.add_trace(go.Scatter(
                x=player_dist['hit_timestamp'],
                y=player_dist['distance_to_ball_m'],
                mode='lines+markers',
                name=f'Player {pid}',
                line=dict(width=2),
                marker=dict(size=6)
            ))
    
    fig.update_layout(
        title="Distance Traveled to Reach Ball Over Time",
        xaxis_title="Timestamp",
        yaxis_title="Distance to Ball (meters)",
        template="plotly_white",
        height=500,
        width=800
    )
    
    fig.show()
    
else:
    print("No distance data available")


Player 1 distance traveled stats:
  Total distance: 113.81m
  Average distance: 2.28m
  Max distance: 6.28m
  Min distance: 0.03m

Player 2 distance traveled stats:
  Total distance: 130.35m
  Average distance: 2.46m
  Max distance: 6.52m
  Min distance: 0.09m



### **SQL Queries**

#### Distance Per Shot

In [97]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

# Query: Distance to Ball per Shot
distance_per_shot_query = f"""
WITH racket_hits AS (
    SELECT *,
           racket_hit_player_id AS player_id,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_x_meter ELSE player_2_x_meter END AS my_x,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_y_meter ELSE player_2_y_meter END AS my_y,
           CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
opponent_last_shots AS (
    SELECT h1.rally_id,
           h1.timestamp AS hit_timestamp,
           h1.player_id,
           h1.opponent_id,
           h2.timestamp AS opponent_last_timestamp,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_x_meter ELSE h2.player_2_x_meter END AS opp_x,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END AS opp_y,
           h1.my_x,
           h1.my_y
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp < h1.timestamp
        AND h2.timestamp = (
            SELECT MAX(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp < h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT player_id,
       opponent_id,
       rally_id,
       hit_timestamp,
       ROUND(SQRT(POWER(my_x - opp_x, 2) + POWER(my_y - opp_y, 2)), 2) AS distance_to_ball_m
FROM opponent_last_shots
ORDER BY player_id, rally_id, hit_timestamp
"""

distance_per_shot_result = pysqldf(distance_per_shot_query)
print("Distance per Shot:")
print(distance_per_shot_result)


Distance per Shot:
     player_id  opponent_id  rally_id  hit_timestamp  distance_to_ball_m
0            1            2         0       2.750000                3.70
1            1            2         0       6.516667                1.26
2            1            2         0       8.866667                1.10
3            1            2         0      13.616667                5.35
4            1            2         0      16.566667                2.90
..         ...          ...       ...            ...                 ...
98           2            1         9     271.233333                1.15
99           2            1        10     288.600000                1.78
100          2            1        10     290.533333                0.65
101          2            1        10     294.666667                0.09
102          2            1        10     297.400000                4.25

[103 rows x 5 columns]


#### Total Distance Per Player

In [101]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""
total_distance_query = f"""
WITH racket_hits AS (
    SELECT *,
           racket_hit_player_id AS player_id,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_x_meter ELSE player_2_x_meter END AS my_x,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_y_meter ELSE player_2_y_meter END AS my_y,
           CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
opponent_last_shots AS (
    SELECT h1.player_id,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_x_meter ELSE h2.player_2_x_meter END AS opp_x,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END AS opp_y,
           h1.my_x,
           h1.my_y
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp < h1.timestamp
        AND h2.timestamp = (
            SELECT MAX(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp < h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT player_id,
       ROUND(SUM(SQRT(POWER(my_x - opp_x, 2) + POWER(my_y - opp_y, 2))), 2) AS total_distance_m
FROM opponent_last_shots
GROUP BY player_id
ORDER BY player_id
"""

total_distance_result = pysqldf(total_distance_query)
print("=== Total Distance per Player ===")
print(total_distance_result)


=== Total Distance per Player ===
   player_id  total_distance_m
0          1            113.81
1          2            130.35


#### Average Distance Per Player: 

In [102]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""
average_distance_query = f"""
WITH racket_hits AS (
    SELECT *,
           racket_hit_player_id AS player_id,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_x_meter ELSE player_2_x_meter END AS my_x,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_y_meter ELSE player_2_y_meter END AS my_y,
           CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
opponent_last_shots AS (
    SELECT h1.player_id,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_x_meter ELSE h2.player_2_x_meter END AS opp_x,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END AS opp_y,
           h1.my_x,
           h1.my_y
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp < h1.timestamp
        AND h2.timestamp = (
            SELECT MAX(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp < h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT player_id,
       ROUND(AVG(SQRT(POWER(my_x - opp_x, 2) + POWER(my_y - opp_y, 2))), 2) AS average_distance_m
FROM opponent_last_shots
GROUP BY player_id
ORDER BY player_id
"""

average_distance_result = pysqldf(average_distance_query)
print("=== Average Distance per Player ===")
print(average_distance_result)


=== Average Distance per Player ===
   player_id  average_distance_m
0          1                2.28
1          2                2.46


#### Min Distance Per Player

In [103]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

min_distance_query = f"""
WITH racket_hits AS (
    SELECT *,
           racket_hit_player_id AS player_id,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_x_meter ELSE player_2_x_meter END AS my_x,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_y_meter ELSE player_2_y_meter END AS my_y,
           CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
opponent_last_shots AS (
    SELECT h1.player_id,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_x_meter ELSE h2.player_2_x_meter END AS opp_x,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END AS opp_y,
           h1.my_x,
           h1.my_y
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp < h1.timestamp
        AND h2.timestamp = (
            SELECT MAX(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp < h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT player_id,
       ROUND(MIN(SQRT(POWER(my_x - opp_x, 2) + POWER(my_y - opp_y, 2))), 2) AS min_distance_m
FROM opponent_last_shots
GROUP BY player_id
ORDER BY player_id
"""

min_distance_result = pysqldf(min_distance_query)
print("=== Min Distance per Player ===")
print(min_distance_result)


=== Min Distance per Player ===
   player_id  min_distance_m
0          1            0.03
1          2            0.09


#### Max Distance Per Player

In [104]:
# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""

max_distance_query = f"""
WITH racket_hits AS (
    SELECT *,
           racket_hit_player_id AS player_id,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_x_meter ELSE player_2_x_meter END AS my_x,
           CASE WHEN racket_hit_player_id = 1 THEN player_1_y_meter ELSE player_2_y_meter END AS my_y,
           CASE WHEN racket_hit_player_id = 1 THEN 2 ELSE 1 END AS opponent_id
    FROM df
    WHERE is_racket_hit = 1 {filter_clause}
),
opponent_last_shots AS (
    SELECT h1.player_id,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_x_meter ELSE h2.player_2_x_meter END AS opp_x,
           CASE WHEN h1.opponent_id = 1 THEN h2.player_1_y_meter ELSE h2.player_2_y_meter END AS opp_y,
           h1.my_x,
           h1.my_y
    FROM racket_hits h1
    LEFT JOIN racket_hits h2
        ON h1.rally_id = h2.rally_id
        AND h2.player_id = h1.opponent_id
        AND h2.timestamp < h1.timestamp
        AND h2.timestamp = (
            SELECT MAX(h3.timestamp)
            FROM racket_hits h3
            WHERE h3.rally_id = h1.rally_id
              AND h3.player_id = h1.opponent_id
              AND h3.timestamp < h1.timestamp
        )
    WHERE h2.timestamp IS NOT NULL
)
SELECT player_id,
       ROUND(MAX(SQRT(POWER(my_x - opp_x, 2) + POWER(my_y - opp_y, 2))), 2) AS max_distance_m
FROM opponent_last_shots
GROUP BY player_id
ORDER BY player_id
"""

max_distance_result = pysqldf(max_distance_query)
print("=== Max Distance per Player ===")
print(max_distance_result)


=== Max Distance per Player ===
   player_id  max_distance_m
0          1            6.28
1          2            6.52


## **8. Straight Shot Quality**

### **SQL Queries**

In [110]:
rally_id = None
player_id = None
start_time = None
end_time = None

# Distance threshold to count as "close to wall" (meters)
wall_threshold = 1.2

# ----------------------
# Filter Racket Hits
# ----------------------
straight_shots = df[
    (df['is_racket_hit'] == True) &
    (df['shot_type'].isin(['straight_drive', 'straight_drop']))
].copy()

if rally_id is not None:
    straight_shots = straight_shots[straight_shots['rally_id'] == rally_id]
if player_id is not None:
    straight_shots = straight_shots[straight_shots['racket_hit_player_id'] == player_id]
if start_time is not None:
    straight_shots = straight_shots[straight_shots['timestamp'] >= start_time]
if end_time is not None:
    straight_shots = straight_shots[straight_shots['timestamp'] <= end_time]

# ----------------------
# Court boundaries
# ----------------------
# Assuming left wall = 0m, right wall = court_width (meters)
court_width = 6.1  # standard singles squash court width

# ----------------------
# Compute distance to nearest wall
# ----------------------
def lateral_distance_to_wall(row):
    x = row['player_1_x_meter'] if row['racket_hit_player_id'] == 1 else row['player_2_x_meter']
    distance_to_left = x
    distance_to_right = court_width - x
    return min(distance_to_left, distance_to_right)

straight_shots['distance_to_wall_m'] = straight_shots.apply(lateral_distance_to_wall, axis=1)
straight_shots['close_to_wall'] = straight_shots['distance_to_wall_m'] <= wall_threshold

# ----------------------
# Aggregate per player
# ----------------------
for pid in [1, 2]:
    player_shots = straight_shots[straight_shots['racket_hit_player_id'] == pid]
    if len(player_shots) > 0:
        total_shots = len(player_shots)
        close_shots = player_shots['close_to_wall'].sum()
        percent_close = (close_shots / total_shots) * 100
        print(f"Player {pid} straight shots: {total_shots}")
        print(f"  Shots close to wall (<={wall_threshold}m): {close_shots}")
        print(f"  Percentage close to wall: {percent_close:.1f}%\n")
    else:
        print(f"Player {pid} has no straight shots in this selection.\n")


Player 1 straight shots: 38
  Shots close to wall (<=1.2m): 7
  Percentage close to wall: 18.4%

Player 2 straight shots: 42
  Shots close to wall (<=1.2m): 8
  Percentage close to wall: 19.0%



In [ ]:
rally_id = None
player_id = None
start_time = None
end_time = None

filter_conditions = []
if rally_id is not None:
    filter_conditions.append(f"rally_id = {rally_id}")
if player_id is not None:
    filter_conditions.append(f"racket_hit_player_id = {player_id}")
if start_time is not None:
    filter_conditions.append(f"timestamp >= '{start_time}'")
if end_time is not None:
    filter_conditions.append(f"timestamp <= '{end_time}'")

filter_clause = " AND " + " AND ".join(filter_conditions) if filter_conditions else ""


# ================================
# SQL: Percentage of Straight Shots Close to Wall (ONLY percentage)
# ================================
close_to_wall_query = f"""
WITH straight_shots AS (
    SELECT
        racket_hit_player_id AS player_id,

        -- Distance to nearest wall using SQLite-safe logic
        CASE 
            WHEN racket_hit_player_id = 1 THEN
                CASE 
                    WHEN player_1_x_meter <= (6.1 - player_1_x_meter) 
                        THEN player_1_x_meter
                    ELSE 
                        (6.1 - player_1_x_meter)
                END
            ELSE
                CASE 
                    WHEN player_2_x_meter <= (6.1 - player_2_x_meter) 
                        THEN player_2_x_meter
                    ELSE 
                        (6.1 - player_2_x_meter)
                END
        END AS distance_to_wall_m

    FROM df
    WHERE is_racket_hit = 1
      AND shot_type IN ('straight_drive', 'straight_drop')
      {filter_clause}
),

flagged AS (
    SELECT
        player_id,
        CASE WHEN distance_to_wall_m <= 1.2 THEN 1 ELSE 0 END AS close_flag
    FROM straight_shots
)

SELECT
    player_id,
    ROUND(100.0 * SUM(close_flag) * 1.0 / COUNT(*), 2) AS pct_close_to_wall
FROM flagged
GROUP BY player_id
ORDER BY player_id
"""

close_to_wall_result = pysqldf(close_to_wall_query)

print("=== Percentage of Straight Shots Close to Wall (<=1.2m) ===")
print(close_to_wall_result)


=== Percentage of Straight Shots Close to Wall (<=1.2m) ===
   player_id  pct_close_to_wall
0          1              18.42
1          2              19.05


## **9. Rally Intensity**

In [122]:
import pandas as pd
import plotly.graph_objects as go

# Filters
rally_id = None
player_id = None
start_time = None
end_time = None

# Copy and filter
df2 = df.copy()
if rally_id is not None:
    df2 = df2[df2["rally_id"] == rally_id]
if player_id is not None:
    df2 = df2[df2["racket_hit_player_id"] == player_id]
if start_time is not None:
    df2 = df2[df2["timestamp"] >= start_time]
if end_time is not None:
    df2 = df2[df2["timestamp"] <= end_time]

# Only racket hits
racket_hits = df2[df2["is_racket_hit"] == True].copy()

if racket_hits.empty:
    print("No racket hits in this selection.")
else:
    # Group by rally
    rally_stats = (
        racket_hits
        .groupby("rally_id")["timestamp"]
        .agg(hits="count", start="min", end="max")
        .reset_index()
    )

    # Compute duration and seconds per hit per rally
    rally_stats["duration_s"] = rally_stats["end"] - rally_stats["start"]
    rally_stats["sec_per_hit"] = rally_stats.apply(
        lambda r: r["duration_s"] / r["hits"] if r["hits"] > 0 else float("nan"),
        axis=1
    )

    print("=== Rally Intensity (Seconds per Hit) per Rally ===")
    print(rally_stats[["rally_id", "hits", "duration_s", "sec_per_hit"]])

    # -----------------------
    # Aggregated metrics across all rallies
    # -----------------------
    avg_sec = rally_stats["sec_per_hit"].mean()
    max_sec = rally_stats["sec_per_hit"].max()
    min_sec = rally_stats["sec_per_hit"].min()

    print("\n=== Rally Intensity Summary Across Game ===")
    print(f"Average seconds per hit: {avg_sec:.2f}")
    print(f"Fastest rally (lowest sec/hit): {min_sec:.2f}")
    print(f"Slowest rally (highest sec/hit): {max_sec:.2f}")

    # -----------------------
    # Line Graph: Seconds per Hit per Rally
    # -----------------------
    fig = go.Figure()

    # Line and markers for each rally
    fig.add_trace(go.Scatter(
        x=rally_stats['rally_id'],
        y=rally_stats['sec_per_hit'],
        mode='lines+markers',
        name='Sec per Hit',
        line=dict(color='blue', width=2),
        marker=dict(size=6)
    ))

    # Average line
    fig.add_trace(go.Scatter(
        x=rally_stats['rally_id'],
        y=[avg_sec]*len(rally_stats),
        mode='lines',
        name='Average',
        line=dict(color='red', width=2, dash='dash')
    ))

    fig.update_layout(
        title='Rally Intensity: Seconds per Hit per Rally',
        xaxis_title='Rally ID',
        yaxis_title='Seconds per Hit',
        template='plotly_white',
        height=500,
        width=800
    )

    fig.show()


=== Rally Intensity (Seconds per Hit) per Rally ===
    rally_id  hits  duration_s  sec_per_hit
0          0    12   16.166667     1.347222
1          1    12   16.466667     1.372222
2          2     9   11.000000     1.222222
3          3     7    7.650000     1.092857
4          4     7    7.850000     1.121429
5          5    12   15.000000     1.250000
6          6    10   13.200000     1.320000
7          7    19   27.883333     1.467544
8          8     6    6.133333     1.022222
9          9    11   16.916667     1.537879
10        10     9   12.383333     1.375926

=== Rally Intensity Summary Across Game ===
Average seconds per hit: 1.28
Fastest rally (lowest sec/hit): 1.02
Slowest rally (highest sec/hit): 1.54


### **SQL Queries**

#### Rally Level Stats

In [118]:
rally_intensity_query = f"""
WITH racket_hits AS (
    SELECT *
    FROM df
    WHERE is_racket_hit = 1
    {filter_clause}
),
rally_stats AS (
    SELECT 
        rally_id,
        COUNT(*) AS hits,
        MIN(timestamp) AS start_time,
        MAX(timestamp) AS end_time,
        (MAX(timestamp) - MIN(timestamp)) AS duration_s,
        CASE 
            WHEN COUNT(*) > 0 THEN (MAX(timestamp) - MIN(timestamp)) * 1.0 / COUNT(*)
            ELSE NULL
        END AS sec_per_hit
    FROM racket_hits
    GROUP BY rally_id
)
SELECT *
FROM rally_stats
ORDER BY rally_id
"""

rally_intensity_result = pysqldf(rally_intensity_query)
print("=== Rally Intensity (Seconds per Hit) per Rally ===")
print(rally_intensity_result)


=== Rally Intensity (Seconds per Hit) per Rally ===
    rally_id  hits  start_time    end_time  duration_s  sec_per_hit
0          0    12    1.466667   17.633333   16.166667     1.347222
1          1    12   32.600000   49.066667   16.466667     1.372222
2          2     9   58.266667   69.266667   11.000000     1.222222
3          3     7   77.200000   84.850000    7.650000     1.092857
4          4     7  102.133333  109.983333    7.850000     1.121429
5          5    12  124.050000  139.050000   15.000000     1.250000
6          6    10  153.600000  166.800000   13.200000     1.320000
7          7    19  193.983333  221.866667   27.883333     1.467544
8          8     6  240.133333  246.266667    6.133333     1.022222
9          9    11  255.650000  272.566667   16.916667     1.537879
10        10     9  286.366667  298.750000   12.383333     1.375926


#### Average Across Rallies

In [119]:
avg_sec_query = f"""
WITH racket_hits AS (
    SELECT *
    FROM df
    WHERE is_racket_hit = 1
    {filter_clause}
),
rally_stats AS (
    SELECT 
        rally_id,
        COUNT(*) AS hits,
        (MAX(timestamp) - MIN(timestamp)) AS duration_s,
        CASE 
            WHEN COUNT(*) > 0 THEN (MAX(timestamp) - MIN(timestamp)) * 1.0 / COUNT(*)
            ELSE NULL
        END AS sec_per_hit
    FROM racket_hits
    GROUP BY rally_id
)
SELECT ROUND(AVG(sec_per_hit), 2) AS avg_sec_per_hit
FROM rally_stats
"""

avg_sec_result = pysqldf(avg_sec_query)
print("Average seconds per hit across game:")
print(avg_sec_result)


Average seconds per hit across game:
   avg_sec_per_hit
0             1.28


#### Slowest Rally Time

In [120]:
max_sec_query = f"""
WITH racket_hits AS (
    SELECT *
    FROM df
    WHERE is_racket_hit = 1
    {filter_clause}
),
rally_stats AS (
    SELECT 
        rally_id,
        COUNT(*) AS hits,
        (MAX(timestamp) - MIN(timestamp)) AS duration_s,
        CASE 
            WHEN COUNT(*) > 0 THEN (MAX(timestamp) - MIN(timestamp)) * 1.0 / COUNT(*)
            ELSE NULL
        END AS sec_per_hit
    FROM racket_hits
    GROUP BY rally_id
)
SELECT ROUND(MAX(sec_per_hit), 2) AS max_sec_per_hit
FROM rally_stats
"""

max_sec_result = pysqldf(max_sec_query)
print("Slowest rally (highest sec/hit):")
print(max_sec_result)


Slowest rally (highest sec/hit):
   max_sec_per_hit
0             1.54


#### Fastest Rally Time

In [121]:
min_sec_query = f"""
WITH racket_hits AS (
    SELECT *
    FROM df
    WHERE is_racket_hit = 1
    {filter_clause}
),
rally_stats AS (
    SELECT 
        rally_id,
        COUNT(*) AS hits,
        (MAX(timestamp) - MIN(timestamp)) AS duration_s,
        CASE 
            WHEN COUNT(*) > 0 THEN (MAX(timestamp) - MIN(timestamp)) * 1.0 / COUNT(*)
            ELSE NULL
        END AS sec_per_hit
    FROM racket_hits
    GROUP BY rally_id
)
SELECT ROUND(MIN(sec_per_hit), 2) AS min_sec_per_hit
FROM rally_stats
"""

min_sec_result = pysqldf(min_sec_query)
print("Fastest rally (lowest sec/hit):")
print(min_sec_result)


Fastest rally (lowest sec/hit):
   min_sec_per_hit
0             1.02
